[![Open In Colab](https://google.com)](https://google.comhamagami/IS2026/blob/main/Demo_26_MC_TD_Qlearning_Comparison.ipynb)

# Demo 26: Monte Carlo, SARSA (TD), and Q-Learning

**挿入位置：p.32–34**　｜　**実行環境：Google Colab**

モデルのない同一迷路で3手法を比較します。公平に**行動価値を使う制御問題**として、MC Control、SARSA（TD制御）、Q-learning（TD制御）の3つを実装します。

**使い方**：ランタイムで「すべてのセルを実行」した後、スライダー／ボタンで条件を変更します。グラフ中の説明は英語、手順と解説は日本語です。CPUで動作します（Demo 28/29 はPyTorchを使用）。

**報酬の添字**：状態 $s_t$ で行動 $a_t$ を実行した結果得る報酬を $r_t$ と呼び、次状態を $s_{t+1}$ とします。経験は $(s_t,a_t,r_t,s_{t+1})$ です。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
try:
    from google.colab import output as colab_output
    colab_output.enable_custom_widget_manager()
except ImportError:
    pass
import ipywidgets as widgets
plt.rcParams.update({'figure.dpi': 105, 'font.size': 11})
np.set_printoptions(precision=3, suppress=True)

In [ ]:
# All maze-based demos share the same 5x5 world and conventions.
# row 0 = top, column 0 = left. The shortest path S->G has 8 steps.
N=5
START=(4,0); GOAL=(0,4)
WALLS=frozenset({(1,1),(2,1),(2,3),(3,3),(3,4)})
ACTIONS=[(-1,0),(1,0),(0,-1),(0,1)]
ACTION_NAMES=['Up','Down','Left','Right']
STATES=[(i,j) for i in range(N) for j in range(N) if (i,j) not in WALLS]
NONTERMINAL=[s for s in STATES if s!=GOAL]

def transition(s,a, step_cost=-0.04, goal_reward=1.0, walls=WALLS):
    if s==GOAL:return GOAL,0.0,True
    di,dj=ACTIONS[a];ns=(s[0]+di,s[1]+dj)
    if not(0<=ns[0]<N and 0<=ns[1]<N) or ns in walls:ns=s
    done=ns==GOAL
    return ns, goal_reward if done else step_cost, done

def render_maze(ax,values=None,policy=None,route=None,title='',walls=WALLS,fmt='.2f'):
    from matplotlib.patches import Rectangle
    ax.set_xlim(0,N);ax.set_ylim(N,0);ax.set_aspect('equal');ax.axis('off')
    valid=[v for s,v in (values or {}).items() if s not in walls and np.isfinite(v)]
    lo=min(valid) if valid else 0;hi=max(valid) if valid else 1
    for i in range(N):
        for j in range(N):
            s=(i,j)
            if s in walls:color='#626B7B'
            elif s==GOAL:color='#FAD9CA'
            elif s==START:color='#CDECFD'
            elif values is not None and s in values and hi>lo:
                color=plt.cm.YlGnBu(0.15+0.58*(values[s]-lo)/(hi-lo))
            else:color='#FFFFFF'
            ax.add_patch(Rectangle((j,i),1,1,facecolor=color,edgecolor='#8392A6',linewidth=1.1))
            if s in walls:continue
            if s==GOAL:label='G'
            elif values is not None and s in values:label=f'{values[s]:{fmt}}'
            elif s==START:label='S'
            else:label=''
            if label:ax.text(j+.5,i+.49,label,ha='center',va='center',color='#102E57',fontsize=9 if values is not None else 13,fontweight='bold')
            if policy is not None and s in policy and s!=GOAL and values is None:
                ax.text(j+.5,i+.72,'↑↓←→'[policy[s]],ha='center',va='center',color='#1763A8',fontsize=13)
    if route and len(route)>1:
        xx=[s[1]+.5 for s in route];yy=[s[0]+.5 for s in route]
        ax.plot(xx,yy,lw=3,color='#F39B2F',marker='o',markersize=4,alpha=.92)
    ax.set_title(title,fontweight='bold',fontsize=12)

In [ ]:
def eps_action(Q,s,epsilon,rng):
    if rng.random()<epsilon:return int(rng.integers(4))
    best=np.flatnonzero(np.isclose(Q[s],np.max(Q[s])))
    return int(rng.choice(best))

def learn_control(algo,episodes=350,epsilon=.2,alpha=.25,gamma=.9,seed=0,cap=120):
    rng=np.random.default_rng(seed);Q={s:np.zeros(4) for s in STATES}
    count={s:np.zeros(4,dtype=int) for s in NONTERMINAL}
    rewards=[];lengths=[];success=[]
    for ep in range(episodes):
        s=START;a=eps_action(Q,s,epsilon,rng);records=[];score=0;done=False
        for t in range(cap):
            ns,r,done=transition(s,a);records.append((s,a,r,ns,done));score+=r
            if algo=='Q-learning':
                target=r + (0 if done else gamma*np.max(Q[ns]));Q[s][a]+=alpha*(target-Q[s][a])
            elif algo=='SARSA':
                na=eps_action(Q,ns,epsilon,rng) if not done else 0
                target=r+(0 if done else gamma*Q[ns][na]);Q[s][a]+=alpha*(target-Q[s][a])
            if done:break
            s=ns
            if algo=='SARSA':a=na
            else:a=eps_action(Q,s,epsilon,rng)
        if algo=='MC control':
            g=0;seen=set()
            for ss,aa,rr,nn,dd in reversed(records):
                g=rr+gamma*g
            # compute each time-specific return, then first-visit for each (state, action)
            rets=np.zeros(len(records));g=0
            for k in range(len(records)-1,-1,-1):g=records[k][2]+gamma*g;rets[k]=g
            for k,(ss,aa,rr,nn,dd) in enumerate(records):
                if (ss,aa) in seen:continue
                seen.add((ss,aa));count[ss][aa]+=1
                Q[ss][aa]+=(rets[k]-Q[ss][aa])/count[ss][aa]
        rewards.append(score);lengths.append(len(records));success.append(int(done))
    return Q,np.asarray(rewards),np.asarray(lengths),np.asarray(success)

def moving_avg(x,window=25):
    return np.convolve(np.asarray(x,dtype=float),np.ones(window)/window,mode='valid')

# Smoke checks
for method in ['MC control','SARSA','Q-learning']:
    q,r,l,done=learn_control(method,episodes=30)
    assert len(r)==30 and np.all(np.isfinite(r))
print('All three algorithms passed the basic check.')

In [ ]:
methods=['MC control','SARSA','Q-learning']
run_w=widgets.Button(description='Compare algorithms',button_style='primary')
E=widgets.IntSlider(value=350,min=100,max=1000,step=50,description='Episodes',continuous_update=False)
eps=widgets.FloatSlider(value=.2,min=0,max=.7,step=.05,description='ε',continuous_update=False)
gam=widgets.FloatSlider(value=.9,min=.5,max=.99,step=.01,description='γ',continuous_update=False)
alpha=widgets.FloatSlider(value=.25,min=.05,max=.9,step=.05,description='TD α',continuous_update=False)
seed=widgets.IntSlider(value=42,min=0,max=100,description='Seed')
out=widgets.Output()
def compare(_):
    with out:
        clear_output(wait=True)
        fig,axs=plt.subplots(1,3,figsize=(15,4.1))
        for i,m in enumerate(methods):
            Q,reward,length,win=learn_control(m,episodes=E.value,epsilon=eps.value,alpha=alpha.value,gamma=gam.value,seed=seed.value)
            k=25;x=np.arange(k,E.value+1)
            axs[0].plot(x,moving_avg(reward,k),label=m)
            axs[1].plot(x,moving_avg(win,k),label=m)
            if i==2:
                greedy={s:int(np.argmax(Q[s])) for s in NONTERMINAL}
                render_maze(axs[2],policy=greedy,title='Q-learning: learned greedy policy')
            print(f'{m:12s}: success(last 50)={win[-50:].mean():.2f}, mean steps(last 50)={length[-50:].mean():.1f}')
        axs[0].set(title='Moving average episode return',xlabel='Episode',ylabel='Return');axs[1].set(title='Moving average success rate',xlabel='Episode',ylabel='Success',ylim=(-.02,1.02))
        for ax in axs[:2]:ax.grid(alpha=.2);ax.legend(fontsize=9)
        plt.tight_layout();plt.show()
run_w.on_click(compare)
display(widgets.VBox([widgets.HBox([E,eps]),widgets.HBox([gam,alpha,seed]),run_w,out]));compare(None)

### 比較の読み方
- **MC control**：エピソード終了後、各$(s,a)$のfirst-visit returnを平均します。
- **SARSA (TD)**：次に*実際に選んだ行動*のQ値を使って1ステップ更新します（on-policy）。
- **Q-learning**：次状態の*最大Q値*を使って1ステップ更新します（off-policy）。

**注意**：初期Q値、ε、学習率、乱数種に依存し、特定の順位になる保証はありません。有限回の比較から「必ずQ学習が速い」などとは結論づけません。

## Try it yourself / 実験課題
- MCが終了時更新、TD系が毎ステップ更新であることを説明する。
- εを変え、探索不足で失敗が続く可能性を観察する。
- SARSAとQ学習の更新目標の違いを説明する。